In [2]:
from pathlib import Path
import pandas as pd

In [35]:
ROOT = Path("/export/home/agletdinov/work/git_projects/fantom.deg-himorna.peaks")
DATA_INITIAL = ROOT / "data_initial"
DATA = ROOT / "data"

In [6]:
to_biomart = DATA_INITIAL / "biomart/mart_export-GRCh38_HGNC.tsv.gz"
to_epifactors_lncrna = DATA_INITIAL / "epifactors/EpiGenes_lncrnas.csv"

In [13]:
df_biomart = pd.read_csv(to_biomart, sep="\t")
df_biomart.shape

(194758, 9)

In [14]:
df_biomart.head(2)

,Gene stable ID,Gene stable ID version,Gene description,Ensembl Canonical,Gene name,Gene type,Gene Synonym,HGNC ID,HGNC symbol
0,ENSG00000210049,ENSG00000210049.1,mitochondrially encoded tRNA-Phe (UUU/C) [Sour...,1.0,MT-TF,Mt_tRNA,MTTF,HGNC:7481,MT-TF
1,ENSG00000210049,ENSG00000210049.1,mitochondrially encoded tRNA-Phe (UUU/C) [Sour...,1.0,MT-TF,Mt_tRNA,TRNF,HGNC:7481,MT-TF


In [32]:
df_biomart_filt = df_biomart.drop_duplicates(["Gene stable ID", "HGNC symbol"])
len(df_biomart_filt)

86368

In [16]:
df_epifactors_lncrna = pd.read_csv(to_epifactors_lncrna, sep=",")
df_epifactors_lncrna.shape

(124, 16)

In [17]:
df_epifactors_lncrna.head(2)

,Id,HGNC approved symbol,Status of entry,HGNC ID,Alternative_names,HGNC approved name,Entrez gene ID,HGNC gene family tag,HGNC family description,Function,PMID for information on function,Target molecule,Target entity,UniProt_ID_target,PMID for information on target,Comment
0,1.0,17A,New,#,17A,#,#,#,#,RNA binding,20888417,RNA,pre-mRNA,#,20888417,Splicing shift from full-length to shorter GAB...
1,2.0,51A,New,#,51A,#,#,#,#,RNA binding,22996644,RNA,pre-mRNA,#,22996644,Splicing shift from A to variant B by formatio...


In [22]:
mask = df_epifactors_lncrna["HGNC approved symbol"].isin(["TUG1"])
df_epifactors_lncrna[mask]

,Id,HGNC approved symbol,Status of entry,HGNC ID,Alternative_names,HGNC approved name,Entrez gene ID,HGNC gene family tag,HGNC family description,Function,PMID for information on function,Target molecule,Target entity,UniProt_ID_target,PMID for information on target,Comment
115,116.0,TUG1,New,26066,"FLJ20618, NCRNA00080, LINC00080",taurine up-regulated 1,55000,#,#,"Histone modifier recruitment, miRNA sponging","29016735, 31858814, 31880241, 30925453","RNA, Protein","miR-299, miR-204-5p, miR-145, miR-132-3p, PRC2",#,"29016735, 31858814, 31880241, 30925453","Acts as a molecular sponge of miR-299, miR-204..."


In [37]:
df_epifactors_with_ensg = pd.merge(left=df_epifactors_lncrna[["HGNC approved symbol","Alternative_names"]], right=df_biomart_filt, left_on="HGNC approved symbol", right_on="HGNC symbol")
len(df_epifactors_with_ensg)

92

In [38]:
df_epifactors_with_ensg.head(2)

,HGNC approved symbol,Alternative_names,Gene stable ID,Gene stable ID version,Gene description,Ensembl Canonical,Gene name,Gene type,Gene Synonym,HGNC ID,HGNC symbol
0,A1BG-AS1,FLJ23569,ENSG00000268895,ENSG00000268895.6,A1BG antisense RNA 1 [Source:HGNC Symbol;Acc:H...,1.0,A1BG-AS1,lncRNA,A1BG-AS,HGNC:37133,A1BG-AS1
1,AIRN,"AIR, NCRNA00088, IGF2RAS, IGF2R-AS1",ENSG00000268257,ENSG00000268257.3,antisense of IGF2R non-protein coding RNA [Sou...,1.0,AIRN,lncRNA,AIR,HGNC:34515,AIRN


In [39]:
len(df_epifactors_with_ensg) / len(df_epifactors_lncrna)

0.7419354838709677

In [36]:
to_himorna_peaks = DATA / "himorna/himorna_peaks-hm_meth-assoc.tsv.gz"
himorna_peaks = pd.read_csv(
        to_himorna_peaks,
        chunksize=100000000,
        sep="\t",
    )
himorna_peaks = pd.concat(himorna_peaks)
print("himorna_peaks: done")

himorna_peaks: done


In [40]:
himorna_peaks.shape

(705240949, 10)

In [41]:
himorna_peaks.head(2)

,hm,lnc_ens,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens
0,H3K27ac,ENSG00000279885,chr18,10441312,10441530,0.50,NaN,NaN,NaN,NaN
1,H3K27ac,ENSG00000277135,chr18,10441312,10441530,0.48,NaN,NaN,NaN,NaN


In [ ]:
himorna_peaks["lnc_name"] = 

In [46]:
himorna_peaks_with_lnc_name = pd.merge(how="left", left=himorna_peaks, right=df_biomart_filt[["Gene stable ID", "HGNC symbol"]], left_on="lnc_ens", right_on="Gene stable ID")
len(himorna_peaks_with_lnc_name)

705242410

In [47]:
himorna_peaks_with_lnc_name.head(2)

,hm,lnc_ens,chrom,start,end,corr,gene_chrom,gene_start,gene_end,gene_ens,Gene stable ID,HGNC symbol
0,H3K27ac,ENSG00000279885,chr18,10441312,10441530,0.50,NaN,NaN,NaN,NaN,ENSG00000279885,NaN
1,H3K27ac,ENSG00000277135,chr18,10441312,10441530,0.48,NaN,NaN,NaN,NaN,ENSG00000277135,NaN


In [50]:
#Пересечение по HGNC
len(set(himorna_peaks_with_lnc_name["HGNC symbol"].unique()) & set(df_epifactors_with_ensg["HGNC approved symbol"].unique()))

64

In [51]:
64 / len(df_epifactors_lncrna)

0.5161290322580645

In [42]:
#Пересечение по ENSG
len(set(himorna_peaks["lnc_ens"].unique()) & set(df_epifactors_with_ensg["Gene stable ID"].unique()))

64

In [43]:
64 / len(df_epifactors_lncrna)

0.5161290322580645

In [1]:
#Пересечение по ENSG
len(set(himorna_peaks["lnc_ens"].unique()) - set(df_epifactors_with_ensg["Gene stable ID"].unique()))

NameError: name 'himorna_peaks' is not defined

In [2]:
#Пересечение по ENSG
set(df_epifactors_with_ensg["Gene stable ID"].unique()) - set(himorna_peaks["lnc_ens"].unique())

NameError: name 'df_epifactors_with_ensg' is not defined